# 01. Tokenization Playground

This notebook compares several tokenization strategies and shows how they change what a model "sees".

We will compare:
- a simple word-level baseline
- a simple character-level baseline
- a raw byte-level baseline
- pretrained subword tokenizers:
  - WordPiece via multilingual BERT
  - byte-level BPE via GPT-2
  - SentencePiece via mT5

The goal is not to declare one tokenizer universally best. The goal is to see where different tokenizers behave well or badly, especially for rare words, apostrophes, diacritics, and morphologically complex forms.


The workshop treats the system as retrieval-first:
- tokenization defines what the model sees
- retrieval quality often controls answer quality
- generators cannot reliably compensate for weak retrieval
- evaluation should include groundedness, citation, abstention, bilingual behavior, and community-review placeholders

Current notebook: `01_tokenization_playground.ipynb`

This notebook shows how segmentation choices affect token counts, fragmentation, vocabulary coverage, and downstream behavior in retrieval or generation.

This is a core workshop notebook.

Workshop sequence:
1. `01_tokenization_playground.ipynb`
2. `02_embeddings_and_similarity.ipynb`
3. `03_retriever_benchmarking_for_rag.ipynb`
4. `04_llm_comparison_in_same_rag_pipeline.ipynb`
5. `05_evaluating_rag_systems.ipynb`
6. `06_optional_lora_or_domain_adaptation.ipynb` (optional)


## Quick concept refresher

- Tokenization turns text into units that models can process.
- Retrieval chooses which passages are available as evidence.
- The retriever selects context; the generator turns context into an answer.
- Evaluation in archive assistants is multi-layered because the system must be useful, grounded, reviewable, and appropriate.


In [ ]:
# Self-contained runtime setup for this notebook.
# Each notebook includes its own seed and device checks so it can run independently in Colab.

import random
import sys
from pathlib import Path

import numpy as np

try:
    import torch
except ImportError:
    torch = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

if torch is not None:
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if (torch is not None and torch.cuda.is_available()) else "cpu"

print(f"Python version: {sys.version.split()[0]}")
print(f"Working directory: {Path.cwd()}")
print(f"Detected device: {DEVICE}")
print(f"Seed set to: {SEED}")

NOTEBOOK_CONTEXT = {
    "seed": SEED,
    "device": DEVICE,
    "notebook": __name__ if "__name__" in globals() else "notebook",
    "framing": "retrieval-first archive assistant workshop",
}

NOTEBOOK_CONTEXT


In [ ]:
# Install tokenizer libraries if needed.
# In Colab, uncomment the next line if transformers is not already available.

# !pip -q install transformers pandas matplotlib seaborn

print("Tokenizer dependencies are ready once transformers, pandas, and matplotlib are installed.")


In [ ]:
# Imports for the notebook.
import re
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from transformers import AutoTokenizer

sns.set_theme(style="whitegrid")


In [ ]:
# Editable examples.
EXAMPLES = {
    "english_plain": "The archive assistant should return the source and say when it is unsure.",
    "english_with_apostrophes": "Community members' place-names shouldn't be normalized without review.",
    "morphologically_rich": "Müzelerdeki kayıtlarımızdanmışsınız gibi konuşmayalım.",
    "diacritics_and_variation": "cafe café co-operate cooperate place-name place name",
    "paste_your_own_here": "Replace this line with your own sentence, orthography, or mixed-language example.",
}

EXAMPLES


In [ ]:
# Load three pretrained tokenizers with different segmentation styles.
# - BERT gives us WordPiece behavior.
# - GPT-2 gives us byte-level BPE behavior.
# - mT5 gives us SentencePiece behavior.

bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")
mt5_tokenizer = AutoTokenizer.from_pretrained("google/mt5-small")

print("Loaded tokenizers:")
print("-", bert_tokenizer.name_or_path)
print("-", gpt2_tokenizer.name_or_path)
print("-", mt5_tokenizer.name_or_path)


In [ ]:
# Baseline tokenizers.

def word_level_tokenize(text: str):
    # Split into alphabetic word-like spans plus punctuation.
    return re.findall(r"\w+|[^\w\s]", text, flags=re.UNICODE)


def character_level_tokenize(text: str):
    # Characters include spaces only if you keep them.
    # We drop spaces here so token counts are easier to compare.
    return [char for char in text if not char.isspace()]


def byte_level_tokenize(text: str):
    # Represent each UTF-8 byte as a compact hex string.
    # This is a transparent baseline for understanding byte segmentation.
    return [f"0x{byte:02x}" for byte in text.encode("utf-8")]


def hf_tokenize(text: str, tokenizer):
    # Convert model ids back into string tokens for display.
    ids = tokenizer(text, add_special_tokens=False)["input_ids"]
    return tokenizer.convert_ids_to_tokens(ids)


TOKENIZER_REGISTRY = {
    "word_baseline": lambda text: word_level_tokenize(text),
    "character_baseline": lambda text: character_level_tokenize(text),
    "byte_baseline": lambda text: byte_level_tokenize(text),
    "wordpiece_mbert": lambda text: hf_tokenize(text, bert_tokenizer),
    "byte_bpe_gpt2": lambda text: hf_tokenize(text, gpt2_tokenizer),
    "sentencepiece_mt5": lambda text: hf_tokenize(text, mt5_tokenizer),
}


In [ ]:
# A compact comparison function.

def compare_tokenizations(text: str, tokenizer_registry=TOKENIZER_REGISTRY, max_preview_tokens: int = 60):
    rows = []
    print("=" * 100)
    print(f"TEXT: {text}")
    print("=" * 100)

    for name, tokenize_fn in tokenizer_registry.items():
        tokens = tokenize_fn(text)
        preview = tokens[:max_preview_tokens]
        print(f"\n{name} | count={len(tokens)}")
        print(" | ".join(preview))
        if len(tokens) > max_preview_tokens:
            print("... [truncated preview]")

        rows.append(
            {
                "tokenizer": name,
                "token_count": len(tokens),
                "avg_token_length_chars": sum(len(token) for token in tokens) / max(len(tokens), 1),
                "unique_tokens": len(set(tokens)),
            }
        )

    return pd.DataFrame(rows).sort_values("token_count")


In [ ]:
# Run the comparison on every example sentence.
# You can replace EXAMPLES with your own dictionary if you want to expand the notebook.

all_summaries = {}

for label, text in EXAMPLES.items():
    print(f"\n\n### Example: {label}")
    summary_df = compare_tokenizations(text)
    display(summary_df)
    all_summaries[label] = summary_df


In [ ]:
# Convert the collected summaries into a single table for plotting.
# Fertility here is a simple educational metric:
# token_count divided by whitespace word count.

plot_rows = []

for label, text in EXAMPLES.items():
    whitespace_word_count = max(len(text.split()), 1)
    for name, tokenize_fn in TOKENIZER_REGISTRY.items():
        tokens = tokenize_fn(text)
        plot_rows.append(
            {
                "example": label,
                "tokenizer": name,
                "token_count": len(tokens),
                "fertility_vs_whitespace_words": len(tokens) / whitespace_word_count,
            }
        )

plot_df = pd.DataFrame(plot_rows)
plot_df.head()


In [ ]:
# Plot token counts across tokenizers for each example.
# This is often the fastest way to make the tokenization differences visible in a workshop.

plt.figure(figsize=(14, 6))
sns.barplot(data=plot_df, x="example", y="token_count", hue="tokenizer")
plt.xticks(rotation=25, ha="right")
plt.title("Token count changes depending on tokenizer choice")
plt.ylabel("Token count")
plt.xlabel("Example")
plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 6))
sns.barplot(data=plot_df, x="example", y="fertility_vs_whitespace_words", hue="tokenizer")
plt.xticks(rotation=25, ha="right")
plt.title("Fertility relative to simple whitespace word count")
plt.ylabel("Tokens per whitespace word")
plt.xlabel("Example")
plt.tight_layout()
plt.show()


In [ ]:
# Inspect how a specific rare or complex word fragments.

candidate_words = [
    "place-names",
    "shouldn't",
    "Müzelerdeki",
    "kayıtlarımızdanmışsınız",
    "café",
]

fragmentation_rows = []

for word in candidate_words:
    for name, tokenize_fn in TOKENIZER_REGISTRY.items():
        tokens = tokenize_fn(word)
        fragmentation_rows.append(
            {
                "word": word,
                "tokenizer": name,
                "token_count": len(tokens),
                "tokens": " | ".join(tokens),
            }
        )

fragmentation_df = pd.DataFrame(fragmentation_rows)
fragmentation_df


In [ ]:
# Vocabulary coverage and OOV behavior matter mostly for closed-vocabulary tokenizers.
# The simple word baseline can demonstrate OOV counts against a tiny toy vocabulary.

toy_vocab = {
    "the",
    "archive",
    "assistant",
    "should",
    "return",
    "source",
    "and",
    "say",
    "when",
    "it",
    "is",
    "unsure",
}

def toy_word_oov_report(text: str, vocab: set[str]):
    words = [token.lower() for token in word_level_tokenize(text) if re.match(r"\w+", token)]
    in_vocab = [word for word in words if word in vocab]
    oov = [word for word in words if word not in vocab]
    return {
        "words": words,
        "in_vocab": in_vocab,
        "oov": oov,
        "oov_rate": len(oov) / max(len(words), 1),
    }


for label, text in EXAMPLES.items():
    report = toy_word_oov_report(text, toy_vocab)
    print(f"\nExample: {label}")
    print("Words:", report["words"])
    print("OOV:", report["oov"])
    print("OOV rate:", round(report["oov_rate"], 3))


## Discussion prompts

- Which tokenizer looked most language-agnostic?
- Which tokenizer fragmented apostrophes or diacritics in surprising ways?
- Which tokenizer produced the longest sequence for the morphologically rich example?
- How might token inflation hurt retrieval chunking or generation cost?
- When might a byte-level strategy help, and when might it become inefficient?


In [ ]:
# Exercise cell: paste your own sentence here and rerun.
# Try one example with punctuation variation and one with a long inflected form.

custom_text = "Paste your own example here."
custom_summary = compare_tokenizations(custom_text)
display(custom_summary)
